# 15 · Robustness Evaluation on Degraded Audio
Evaluate candidate SVM and Random Forest classifiers under realistic acoustic degradation (low-pass filtering at 4 kHz, additive white noise, mild clipping) across 3 reproducible seeds on held-out test Fold 10.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.preprocessing.audio import load_fixed, extract_features, SR
from src.preprocessing.degrade import degrade

print('Successfully imported audio preprocessing, degradation functions, and sklearn metrics')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported audio preprocessing, degradation functions, and sklearn metrics


In [2]:
# Cell 2 – verify sha256 hashes of inputs, load data, test set, and candidate models
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'
svm_model_path = REPO_ROOT / 'models/svm_candidate.joblib'
rf_model_path = REPO_ROOT / 'models/rf_candidate.joblib'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)
split_sha = compute_sha256(split_path)
svm_sha = compute_sha256(svm_model_path)
rf_sha = compute_sha256(rf_model_path)

print(f'labels.csv sha256            : {labels_sha}')
print(f'audio_features.npy sha256    : {features_sha}')
print(f'split_indices.npz sha256     : {split_sha}')
print(f'svm_candidate.joblib sha256  : {svm_sha}')
print(f'rf_candidate.joblib sha256   : {rf_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'
EXPECTED_SPLIT_SHA  = '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b'
EXPECTED_SVM_SHA    = '3e798d576f2d30e7bb4ae19345879598527d07f33385d9b9f1529d433de3d9e2'
EXPECTED_RF_SHA     = '40f38b939c23930f621fa3ba66de0ce06c9f2fec8da70aeaf9b87ad8ed6d6113'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'
assert split_sha == EXPECTED_SPLIT_SHA, f'split_indices.npz SHA mismatch: {split_sha}'
assert svm_sha == EXPECTED_SVM_SHA, f'svm_candidate.joblib SHA mismatch: {svm_sha}'
assert rf_sha == EXPECTED_RF_SHA, f'rf_candidate.joblib SHA mismatch: {rf_sha}'

labels = pd.read_csv(labels_path)
audio_features = np.load(features_path)

with np.load(split_path) as split_data:
    test_idx = split_data['test_idx']

X_test_clean = audio_features[test_idx]
y_test = labels.loc[test_idx, 'app_class'].to_numpy()

svm_model = joblib.load(svm_model_path)
rf_model = joblib.load(rf_model_path)

print(f'Loaded test_idx length      : {len(test_idx)}')
print(f'X_test_clean shape          : {X_test_clean.shape}')
print(f'y_test shape                : {y_test.shape}')

assert len(test_idx) == 605, f'Expected 605 test rows, got {len(test_idx)}'
assert X_test_clean.shape == (605, 128), f'Expected X_test_clean (605, 128), got {X_test_clean.shape}'
assert len(y_test) == 605, f'Expected 605 in y_test, got {len(y_test)}'
print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256            : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256    : ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
split_indices.npz sha256     : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
svm_candidate.joblib sha256  : 3e798d576f2d30e7bb4ae19345879598527d07f33385d9b9f1529d433de3d9e2
rf_candidate.joblib sha256   : 40f38b939c23930f621fa3ba66de0ce06c9f2fec8da70aeaf9b87ad8ed6d6113


Loaded test_idx length      : 605
X_test_clean shape          : (605, 128)
y_test shape                : (605,)
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – generate degraded features for fold 10 clips across 3 seeds [0, 1, 2]
X_test_deg = {}

t_start = time.perf_counter()
for seed in [0, 1, 2]:
    feats_seed = []
    for row_index, idx in enumerate(tqdm(test_idx, desc=f'Degrading fold 10 (seed {seed})', mininterval=2.0)):
        path = REPO_ROOT / labels.loc[idx, 'path']
        y = load_fixed(path)
        rng = np.random.default_rng(seed * 100000 + row_index)
        yd = degrade(y, SR, rng)
        feature = extract_features(yd)
        feats_seed.append(feature)
        
    arr = np.array(feats_seed, dtype=np.float32)
    assert arr.shape == (605, 128), f'Seed {seed}: Expected shape (605, 128), got {arr.shape}'
    assert arr.dtype == np.float32, f'Seed {seed}: Expected dtype float32, got {arr.dtype}'
    assert not np.isnan(arr).any(), f'Seed {seed}: NaN detected'
    assert not np.isinf(arr).any(), f'Seed {seed}: Inf detected'
    X_test_deg[seed] = arr

t_elapsed = time.perf_counter() - t_start
print(f'Generated degraded features for 3 seeds in {t_elapsed:.2f} s ({t_elapsed/60:.2f} min)')
for seed in [0, 1, 2]:
    print(f'X_test_deg[{seed}] shape: {X_test_deg[seed].shape}, dtype: {X_test_deg[seed].dtype}')
print('Cell 3 asserts passed ✓')

Degrading fold 10 (seed 0):   0%|          | 0/605 [00:00<?, ?it/s]

Degrading fold 10 (seed 0):   0%|          | 1/605 [00:07<1:15:28,  7.50s/it]

Degrading fold 10 (seed 0):   7%|▋         | 41/605 [00:09<01:40,  5.61it/s] 

Degrading fold 10 (seed 0):  14%|█▍        | 84/605 [00:11<00:51, 10.07it/s]

Degrading fold 10 (seed 0):  20%|██        | 124/605 [00:13<00:37, 12.87it/s]

Degrading fold 10 (seed 0):  26%|██▋       | 160/605 [00:15<00:31, 14.13it/s]

... (31 outputs left out)

Degrading fold 10 (seed 2):  89%|████████▉ | 539/605 [00:18<00:02, 29.02it/s]

Degrading fold 10 (seed 2): 100%|█████████▉| 603/605 [00:20<00:00, 29.91it/s]

Degrading fold 10 (seed 2): 100%|██████████| 605/605 [00:21<00:00, 28.80it/s]

Generated degraded features for 3 seeds in 80.84 s (1.35 min)
X_test_deg[0] shape: (605, 128), dtype: float32
X_test_deg[1] shape: (605, 128), dtype: float32
X_test_deg[2] shape: (605, 128), dtype: float32
Cell 3 asserts passed ✓


In [4]:
# Cell 4 – recompute clean features for the 605 fold-10 clips and assert np.allclose
clean_recomputed = []
for idx in tqdm(test_idx, desc='Recomputing clean fold-10 features', mininterval=2.0):
    path = REPO_ROOT / labels.loc[idx, 'path']
    clean_recomputed.append(extract_features(load_fixed(path)))

clean_arr = np.array(clean_recomputed, dtype=np.float32)
max_clean_diff = float(np.max(np.abs(clean_arr - X_test_clean)))
print(f'Max absolute difference vs X_test_clean: {max_clean_diff:.10e}')

assert np.allclose(clean_arr, X_test_clean, atol=1e-5), f'Clean features mismatch: max diff = {max_clean_diff}'
print('Cell 4 asserts passed: clean feature extraction verified ✓')

Recomputing clean fold-10 features:   0%|          | 0/605 [00:00<?, ?it/s]

Recomputing clean fold-10 features:  14%|█▎        | 83/605 [00:02<00:12, 41.37it/s]

Recomputing clean fold-10 features:  27%|██▋       | 166/605 [00:04<00:11, 37.53it/s]

Recomputing clean fold-10 features:  40%|████      | 242/605 [00:07<00:10, 33.14it/s]

Recomputing clean fold-10 features:  54%|█████▎    | 324/605 [00:09<00:07, 35.87it/s]

Recomputing clean fold-10 features:  66%|██████▌   | 400/605 [00:11<00:05, 36.51it/s]

Recomputing clean fold-10 features:  82%|████████▏ | 499/605 [00:13<00:02, 40.70it/s]

Recomputing clean fold-10 features:  98%|█████████▊| 591/605 [00:15<00:00, 42.35it/s]

Recomputing clean fold-10 features: 100%|██████████| 605/605 [00:15<00:00, 39.68it/s]

Max absolute difference vs X_test_clean: 0.0000000000e+00
Cell 4 asserts passed: clean feature extraction verified ✓


In [5]:
# Cell 5 – evaluate accuracy and macro-F1 for SVM and RF across clean and 3 degraded seeds
models = {
    'SVM': svm_model,
    'RF': rf_model
}

conditions = {
    'clean': X_test_clean,
    'deg_seed0': X_test_deg[0],
    'deg_seed1': X_test_deg[1],
    'deg_seed2': X_test_deg[2]
}

eval_records = []
predictions_store = {}

for m_name, model in models.items():
    predictions_store[m_name] = {}
    for c_name, X_eval in conditions.items():
        y_pred = model.predict(X_eval)
        predictions_store[m_name][c_name] = y_pred
        acc = float(accuracy_score(y_test, y_pred))
        mf1 = float(f1_score(y_test, y_pred, average='macro'))
        eval_records.append({
            'model': m_name,
            'condition': c_name,
            'accuracy': acc,
            'macro_f1': mf1
        })

df_eval_summary = pd.DataFrame(eval_records)
print('=== Clean vs Degraded Performance Summary ===')
print(df_eval_summary.round(4).to_string(index=False))

print('\n=== Degraded Statistics and Accuracy Drop ===')
for m_name in ['SVM', 'RF']:
    m_df = df_eval_summary[df_eval_summary['model'] == m_name]
    clean_acc = float(m_df[m_df['condition'] == 'clean']['accuracy'].iloc[0])
    clean_mf1 = float(m_df[m_df['condition'] == 'clean']['macro_f1'].iloc[0])
    
    deg_rows = m_df[m_df['condition'].str.startswith('deg_seed')]
    deg_acc_mean = float(deg_rows['accuracy'].mean())
    deg_acc_std = float(deg_rows['accuracy'].std())
    deg_mf1_mean = float(deg_rows['macro_f1'].mean())
    deg_mf1_std = float(deg_rows['macro_f1'].std())
    
    acc_drop = clean_acc - deg_acc_mean
    acc_drop_pts = acc_drop * 100.0
    
    print(f'Model: {m_name:4s}')
    print(f'  Clean Accuracy       : {clean_acc:.4f} ({clean_acc*100:.2f}%)')
    print(f'  Degraded Acc (mean)  : {deg_acc_mean:.4f} (+/- {deg_acc_std:.4f})')
    print(f'  Accuracy Drop        : {acc_drop:.4f} ({acc_drop_pts:.2f} percentage points)')
    print(f'  Clean Macro-F1       : {clean_mf1:.4f}')
    print(f'  Degraded Macro-F1    : {deg_mf1_mean:.4f} (+/- {deg_mf1_std:.4f})\n')

print('Cell 5 completed ✓')

=== Clean vs Degraded Performance Summary ===
model condition  accuracy  macro_f1
  SVM     clean    0.7967    0.7780
  SVM deg_seed0    0.5041    0.4682
  SVM deg_seed1    0.5240    0.4958
  SVM deg_seed2    0.5223    0.4826
   RF     clean    0.7521    0.7178
   RF deg_seed0    0.4496    0.3997
   RF deg_seed1    0.4678    0.4169
   RF deg_seed2    0.4810    0.4245

=== Degraded Statistics and Accuracy Drop ===
Model: SVM 
  Clean Accuracy       : 0.7967 (79.67%)
  Degraded Acc (mean)  : 0.5168 (+/- 0.0110)
  Accuracy Drop        : 0.2799 (27.99 percentage points)
  Clean Macro-F1       : 0.7780
  Degraded Macro-F1    : 0.4822 (+/- 0.0138)

Model: RF  
  Clean Accuracy       : 0.7521 (75.21%)
  Degraded Acc (mean)  : 0.4661 (+/- 0.0158)
  Accuracy Drop        : 0.2860 (28.60 percentage points)
  Clean Macro-F1       : 0.7178
  Degraded Macro-F1    : 0.4137 (+/- 0.0127)

Cell 5 completed ✓


In [6]:
# Cell 6 – per-class F1 for clean vs degraded, SVM confusion matrix on seed 0, and worst F1 drop
classes = sorted(np.unique(y_test))

per_class_summary = []
worst_class_report = {}

for m_name in ['SVM', 'RF']:
    clean_pred = predictions_store[m_name]['clean']
    rep_clean = classification_report(y_test, clean_pred, output_dict=True)
    
    deg_f1_by_seed = {cls: [] for cls in classes}
    for s in [0, 1, 2]:
        pred_deg = predictions_store[m_name][f'deg_seed{s}']
        rep_deg = classification_report(y_test, pred_deg, output_dict=True)
        for cls in classes:
            deg_f1_by_seed[cls].append(rep_deg[cls]['f1-score'])
            
    class_drops = {}
    for cls in classes:
        c_f1 = rep_clean[cls]['f1-score']
        d_f1_mean = float(np.mean(deg_f1_by_seed[cls]))
        drop = c_f1 - d_f1_mean
        class_drops[cls] = drop
        per_class_summary.append({
            'model': m_name,
            'app_class': cls,
            'clean_f1': c_f1,
            'degraded_mean_f1': d_f1_mean,
            'f1_drop': drop
        })
    worst_cls = max(class_drops, key=class_drops.get)
    worst_class_report[m_name] = (worst_cls, class_drops[worst_cls])

df_per_class = pd.DataFrame(per_class_summary)
print('=== Per-class F1: Clean vs Degraded (mean over seeds) ===')
print(df_per_class.round(4).to_string(index=False))

print('\n=== Class with Largest F1 Loss ===')
for m_name, (w_cls, w_drop) in worst_class_report.items():
    print(f'{m_name}: Class "{w_cls}" lost the most F1 (drop = {w_drop:.4f})')

# Confusion matrix of SVM on seed 0 degraded features
cm_svm_deg0 = confusion_matrix(y_test, predictions_store['SVM']['deg_seed0'], labels=classes)
cm_df = pd.DataFrame(cm_svm_deg0, index=classes, columns=classes)
cm_df.index.name = 'True'
cm_df.columns.name = 'Predicted'

print('\n=== SVM Confusion Matrix (Degraded Seed 0) ===')
print(cm_df.to_string())

fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
cm_fig_path = fig_dir / 'svm_confusion_degraded.png'

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm_deg0, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title('SVM RBF - Confusion Matrix (Degraded Seed 0, Test Fold 10)')
plt.tight_layout()
plt.savefig(cm_fig_path, dpi=150)
plt.close(fig)

print(f'\nSaved degraded confusion matrix plot to: {cm_fig_path}')
print('Cell 6 completed ✓')

=== Per-class F1: Clean vs Degraded (mean over seeds) ===
model         app_class  clean_f1  degraded_mean_f1  f1_drop
  SVM      construction    0.8500            0.5584   0.2916
  SVM              hvac    0.6849            0.5160   0.1689
  SVM music_loudspeaker    0.8406            0.1862   0.6543
  SVM             siren    0.6667            0.5702   0.0965
  SVM      traffic_horn    0.8477            0.5802   0.2676
   RF      construction    0.8321            0.5779   0.2542
   RF              hvac    0.7174            0.5813   0.1361
   RF music_loudspeaker    0.7531            0.2295   0.5237
   RF             siren    0.5323            0.3043   0.2280
   RF      traffic_horn    0.7540            0.3755   0.3785

=== Class with Largest F1 Loss ===
SVM: Class "music_loudspeaker" lost the most F1 (drop = 0.6543)
RF: Class "music_loudspeaker" lost the most F1 (drop = 0.5237)

=== SVM Confusion Matrix (Degraded Seed 0) ===
Predicted          construction  hvac  music_loudspeaker  si


Saved degraded confusion matrix plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\svm_confusion_degraded.png
Cell 6 completed ✓


In [7]:
# Cell 7 – save reports/degraded_eval.csv and print sha256
csv_rows = []

for m_name in ['SVM', 'RF']:
    for c_name in ['clean', 'deg_seed0', 'deg_seed1', 'deg_seed2']:
        y_pred = predictions_store[m_name][c_name]
        acc = float(accuracy_score(y_test, y_pred))
        mf1 = float(f1_score(y_test, y_pred, average='macro'))
        rep = classification_report(y_test, y_pred, output_dict=True)
        
        row_dict = {
            'model': m_name,
            'condition': c_name,
            'accuracy': acc,
            'macro_f1': mf1
        }
        for cls in classes:
            row_dict[f'f1_{cls}'] = rep[cls]['f1-score']
        csv_rows.append(row_dict)

df_csv = pd.DataFrame(csv_rows)
out_csv_path = REPO_ROOT / 'reports/degraded_eval.csv'
df_csv.to_csv(out_csv_path, index=False)
print(f'Saved degraded evaluation table to: {out_csv_path}')

print('\n=== Degraded Evaluation Export Table ===')
print(df_csv.round(4).to_string(index=False))

eval_csv_sha = compute_sha256(out_csv_path)
print(f'\nreports/degraded_eval.csv sha256: {eval_csv_sha}')
print('Cell 7 completed ✓')

Saved degraded evaluation table to: C:\Users\LENOVO\Desktop\CityPulse\reports\degraded_eval.csv

=== Degraded Evaluation Export Table ===
model condition  accuracy  macro_f1  f1_construction  f1_hvac  f1_music_loudspeaker  f1_siren  f1_traffic_horn
  SVM     clean    0.7967    0.7780           0.8500   0.6849                0.8406    0.6667           0.8477
  SVM deg_seed0    0.5041    0.4682           0.5513   0.4948                0.1525    0.5821           0.5603
  SVM deg_seed1    0.5240    0.4958           0.5662   0.5071                0.2381    0.5802           0.5872
  SVM deg_seed2    0.5223    0.4826           0.5577   0.5461                0.1681    0.5484           0.5929
   RF     clean    0.7521    0.7178           0.8321   0.7174                0.7531    0.5323           0.7540
   RF deg_seed0    0.4496    0.3997           0.5642   0.5621                0.1951    0.3436           0.3333
   RF deg_seed1    0.4678    0.4169           0.5758   0.5871                0.2697  